# 🔤 Notebook 02 — NLP: Embeddings y Similitud de Narrativas
## FraudIA Claims · hackIAthon 2026 · Reto Aseguradora del Sur

Este notebook implementa el análisis NLP completo de las narrativas de siniestros:
- Vectorización TF-IDF con n-gramas
- Detección de narrativas clonadas (Regla RF-07: similitud > 85%)
- Visualización de clusters semánticos
- Ranking de alertas por similitud textual


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

COLORS = {'rojo': '#BA1A1A', 'amarillo': '#FFB800', 'verde': '#00A344', 'azul': '#002662'}

# Carga de datos
df = pd.read_csv('../data/synthetic/siniestros_scored.csv')
df['etiqueta_fraude_simulada'] = df['etiqueta_fraude_simulada'].astype(int)
print(f"✅ Dataset cargado: {len(df)} siniestros")
print(f"   Narrativas disponibles: {df['descripcion'].notna().sum()}")


## 1. Preprocesamiento y Vectorización TF-IDF

In [ ]:
# Preprocesamiento básico del texto
def preprocesar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower().strip()
    # Remover caracteres especiales manteniendo tildes (español)
    import re
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['descripcion_clean'] = df['descripcion'].apply(preprocesar_texto)

# Vectorización TF-IDF con n-gramas 1 y 2
vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2),      # unigramas y bigramas
    max_features=500,
    sublinear_tf=True,       # escala logarítmica para TF
)
tfidf_matrix = vectorizer.fit_transform(df['descripcion_clean'])

print(f"✅ Matriz TF-IDF construida: {tfidf_matrix.shape}")
print(f"   Vocabulario: {len(vectorizer.vocabulary_)} términos")
print(f"   Sparsidad  : {(1 - tfidf_matrix.nnz / np.prod(tfidf_matrix.shape))*100:.1f}%")

# Top 20 términos más frecuentes
feature_names = vectorizer.get_feature_names_out()
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top20_idx = mean_tfidf.argsort()[::-1][:20]
print(f"\n📝 Top 20 términos más relevantes:")
for i, idx in enumerate(top20_idx):
    print(f"   {i+1:2}. {feature_names[idx]:<35} peso: {mean_tfidf[idx]:.4f}")


## 2. Detección de Narrativas Clonadas — Regla RF-07

In [ ]:
# Calcular similitud coseno entre todos los pares
sim_matrix = cosine_similarity(tfidf_matrix)
np.fill_diagonal(sim_matrix, 0)  # eliminar auto-similitud

# Extraer pares con similitud alta
pares_clonados = []
visitados = set()

for i in range(len(df)):
    j = int(sim_matrix[i].argmax())
    sim = float(sim_matrix[i, j])
    
    par = tuple(sorted([i, j]))
    if par in visitados or sim < 0.50:
        continue
    visitados.add(par)
    
    pares_clonados.append({
        'sin_1':       df.iloc[i]['id_siniestro'],
        'sin_2':       df.iloc[j]['id_siniestro'],
        'similitud':   round(sim, 4),
        'nivel_alerta': 'ROJO' if sim > 0.85 else 'AMARILLO' if sim > 0.70 else 'VERDE',
        'fraude_1':    df.iloc[i]['etiqueta_fraude_simulada'],
        'fraude_2':    df.iloc[j]['etiqueta_fraude_simulada'],
        'descripcion': df.iloc[i]['descripcion'][:80] + '...',
    })

df_pares = pd.DataFrame(pares_clonados).sort_values('similitud', ascending=False)

print("=" * 60)
print(f"  ANÁLISIS RF-07 — NARRATIVAS CLONADAS")
print("=" * 60)
print(f"  Pares con similitud > 85% (ROJO)    : {(df_pares['nivel_alerta']=='ROJO').sum()}")
print(f"  Pares con similitud 70-85% (AMARILLO): {(df_pares['nivel_alerta']=='AMARILLO').sum()}")
print(f"  Pares con similitud 50-70% (VERDE)  : {(df_pares['nivel_alerta']=='VERDE').sum()}")
print("\nTop 10 pares más similares:")
df_pares.head(10)[['sin_1','sin_2','similitud','nivel_alerta','fraude_1','fraude_2']].to_string(index=False)


## 3. Visualización — Mapa de Calor de Similitud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap top 20 siniestros más sospechosos
top20_sin = df.nlargest(20, 'score_riesgo')['id_siniestro'].tolist()
idx_top20  = [df[df['id_siniestro']==s].index[0] for s in top20_sin]
sim_top20  = sim_matrix[np.ix_(idx_top20, idx_top20)]

im = axes[0].imshow(sim_top20, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
axes[0].set_xticks(range(20))
axes[0].set_yticks(range(20))
labels_short = [s.replace('SIN-0','S') for s in top20_sin]
axes[0].set_xticklabels(labels_short, rotation=90, fontsize=7)
axes[0].set_yticklabels(labels_short, fontsize=7)
axes[0].set_title('Similitud Textual — Top 20 Siniestros de Mayor Riesgo', fontweight='bold', fontsize=11)
plt.colorbar(im, ax=axes[0], label='Similitud Coseno')

# Distribución de similitudes
sims_all = df_pares['similitud'].values
axes[1].hist(sims_all[sims_all > 0.85], bins=15, color=COLORS['rojo'],  alpha=0.8, label='ROJO  (>85%)')
axes[1].hist(sims_all[(sims_all > 0.70) & (sims_all <= 0.85)], bins=10,
             color=COLORS['amarillo'], alpha=0.8, label='AMARILLO (70-85%)')
axes[1].hist(sims_all[sims_all <= 0.70], bins=15, color=COLORS['verde'], alpha=0.8, label='VERDE (<70%)')
axes[1].axvline(x=0.85, color='black', linestyle='--', linewidth=2, label='Umbral RF-07')
axes[1].set_title('Distribución de Similitudes entre Narrativas', fontweight='bold', fontsize=11)
axes[1].set_xlabel('Similitud Coseno')
axes[1].set_ylabel('Cantidad de Pares')
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Reducción Dimensional — Visualización 2D de Narrativas

In [ ]:
# Reducción a 2D con SVD truncado (LSA)
svd = TruncatedSVD(n_components=2, random_state=42)
coords_2d = svd.fit_transform(tfidf_matrix)

print(f"✅ SVD completado")
print(f"   Varianza explicada: {svd.explained_variance_ratio_.sum()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot por nivel de riesgo
colores_nivel = {'VERDE': COLORS['verde'], 'ROJO': COLORS['rojo'], 'AMARILLO': COLORS['amarillo']}
for nivel, color in colores_nivel.items():
    mask = df['nivel_riesgo'] == nivel
    axes[0].scatter(coords_2d[mask, 0], coords_2d[mask, 1],
                   c=color, label=nivel, alpha=0.7, s=40, edgecolors='white', linewidth=0.3)
axes[0].set_title('Clusters Semánticos por Nivel de Riesgo (LSA 2D)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Componente 1')
axes[0].set_ylabel('Componente 2')
axes[0].legend(title='Nivel Riesgo')

# Plot por etiqueta de fraude
for etiq, color, label in [(0, COLORS['verde'], 'Normal'), (1, COLORS['rojo'], 'Fraude Simulado')]:
    mask = df['etiqueta_fraude_simulada'] == etiq
    axes[1].scatter(coords_2d[mask, 0], coords_2d[mask, 1],
                   c=color, label=label, alpha=0.7, s=40, edgecolors='white', linewidth=0.3)
axes[1].set_title('Clusters Semánticos por Etiqueta Fraude (LSA 2D)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Componente 1')
axes[1].set_ylabel('Componente 2')
axes[1].legend(title='Etiqueta')

plt.suptitle('FraudIA · Análisis de Similitud Semántica de Narrativas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 5. Palabras Clave Asociadas a Fraude

In [ ]:
# Términos con mayor diferencia entre fraude y normal
feature_names = vectorizer.get_feature_names_out()
tfidf_fraude  = np.asarray(tfidf_matrix[df['etiqueta_fraude_simulada']==1].mean(axis=0)).flatten()
tfidf_normal  = np.asarray(tfidf_matrix[df['etiqueta_fraude_simulada']==0].mean(axis=0)).flatten()
diferencia    = tfidf_fraude - tfidf_normal

# Top 15 términos más asociados a fraude y a normalidad
top_fraude = np.argsort(diferencia)[::-1][:15]
top_normal = np.argsort(diferencia)[:15]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh([feature_names[i] for i in top_fraude[::-1]],
             [diferencia[i] for i in top_fraude[::-1]],
             color=COLORS['rojo'], alpha=0.8)
axes[0].set_title('Términos más asociados a FRAUDE SIMULADO', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Diferencia TF-IDF (Fraude − Normal)')

axes[1].barh([feature_names[i] for i in top_normal],
             [abs(diferencia[i]) for i in top_normal],
             color=COLORS['verde'], alpha=0.8)
axes[1].set_title('Términos más asociados a CASO NORMAL', fontweight='bold', fontsize=11)
axes[1].set_xlabel('Diferencia TF-IDF (Normal − Fraude)')

plt.tight_layout()
plt.show()

print("\n✅ HALLAZGOS NLP:")
print("  1. La regla RF-07 detecta narrativas con >85% similitud")
print(f"  2. {(df_pares['nivel_alerta']=='ROJO').sum()} pares de siniestros comparten narrativas casi idénticas")
print("  3. Los casos de fraude usan vocabulario relacionado con 'zona sin cámaras', 'sin testigos'")
print("  4. Los casos normales presentan mayor variedad léxica y descripciones más detalladas")
